In [ ]:
# Find bound changes that can restore production of a specific compound
import cobra,json,os,sys,re,pandas as pd, io, requests
from collections import OrderedDict
from copy import deepcopy

# keep going up until the "pycore" folder is found
wd = os.getcwd()
os.chdir('..')
base_path = os.getcwd()
sys.path.append(base_path)
from pycore import *

os.chdir(wd)

/Users/ejm6426/Documents/pkRBA/pycore/../build_model/input/iIsor850_RBA.json


In [ ]:
compound_to_target = "5aop_e"
rxns_to_avoid_changing = {"EX_5aop_e"}
from cobra.flux_analysis import gapfill
with model as m:
	objective_rxns = [r for r in m.reactions if r.objective_coefficient != 0]
	# add compound_to_target as reactant of objective reactions, to require it to be produced in the gapfilling solution
	for r in objective_rxns:
		if compound_to_target not in [met.id for met in r.reactants]:
			met = m.metabolites.get_by_id(compound_to_target)
			r.add_metabolites({met: -1})
	universal = cobra.Model("universal_reactions")
	for r in m.reactions:
		if r.bounds == (-1000, 1000) or r.id in set([rxn.id for rxn in objective_rxns]) | set([rxn.id for rxn in m.exchanges]) | rxns_to_avoid_changing:
			continue
		universal.add_reactions([r.copy()])
		universal.reactions.get_by_id(r.id).bounds = (-1000, 1000)
		# change the id of the reaction in the universal model to avoid conflicts with the original model
		universal.reactions.get_by_id(r.id).id = r.id + "_universal"
	gapfill_solution = gapfill(m, universal)
	for reaction in gapfill_solution[0]:
		print(reaction.id)

5AOPtps_e_universal
